<a href="https://colab.research.google.com/github/francescopassante/GETMeshClassificator/blob/main/GET/src/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content

/content


In [2]:
%rm -rf GETMeshClassificator/
!git clone https://github.com/francescopassante/GETMeshClassificator.git
%cd /content

Cloning into 'GETMeshClassificator'...
remote: Enumerating objects: 321, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 321 (delta 98), reused 263 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (321/321), 291.82 KiB | 3.10 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content


In [5]:
# Download dataset with gdown:
!gdown 1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE
!unzip -q SHREC11_200NEIGH.zip

Downloading...
From (original): https://drive.google.com/uc?id=1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE
From (redirected): https://drive.google.com/uc?id=1tkg25dLF-sgbNinNv8q6FCa-Z87d53kE&confirm=t&uuid=1753ed2c-08f2-4d4f-ba52-c311cc30ff15
To: /content/SHREC11_200NEIGH.zip
100% 2.41G/2.41G [00:24<00:00, 96.9MB/s]


In [30]:
%cd /content
!unzip -q SHREC11_50NEIGH.zip

/content


In [33]:
%cd GETMeshClassificator/GET/src

[Errno 2] No such file or directory: 'GETMeshClassificator/GET/src'
/content/GETMeshClassificator/GET/src


In [58]:
import GET
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

train_loader, test_loader = GET.load_data(
        mesh_directory="../../../SHREC11_200/processed/",
        labels_file="../../../SHREC11_200/classes.txt",
        N=9,
        train_percent=0.01,
)
print(len(train_loader), len(test_loader))

5 594


In [65]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GET.GETClassifier(N=9, channels=12, heads = 2, out_classes=30).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.1)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model has {num_params} parameters")

Model has 9114 parameters


In [37]:
def train(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=1,
    accumulation_steps=16,
):
    model.train()
    loss_hist = []

    for epoch in range(epochs):
        total_loss = 0.0

        # Start with zeroed gradients for the first accumulation block
        optimizer.zero_grad()

        for i, mesh in enumerate(tqdm(dataloader, desc=f"Epoch {epoch + 1}/{epochs}")):
            x = mesh["x"].to(device).squeeze(0)
            neighbors = mesh["neighbors"].to(device).squeeze(0)
            mask = mesh["mask"].to(device).squeeze(0)
            parallel_transport_matrices = (
                mesh["parallel_transport_matrices"].to(device).squeeze(0)
            )
            rel_pos_u = mesh["rel_pos"].to(device).squeeze(0)
            labels = mesh["label"].to(device).long().squeeze(0)

            # Forward
            outputs = model(x, neighbors, mask, parallel_transport_matrices, rel_pos_u)
            # Print the index of dimension with max output
            # print("mesh file: ", mesh["filenumber"])
            # print("predicted label: ", torch.argmax(outputs).item())
            # print("True label: ", labels.item())
            raw_loss = criterion(outputs.unsqueeze(0), labels.unsqueeze(0))

            if torch.isnan(raw_loss):
                print("NAN LOSS")

            # Scale the loss for gradient accumulation
            scaled_loss = raw_loss / accumulation_steps
            scaled_loss.backward()

            # Accumulate the raw (unscaled) loss for logging
            total_loss += raw_loss.item()

            # Step and zero gradients at the accumulation boundary (or final batch)
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
                #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

        # Average epoch loss (uses unscaled batch losses)
        epoch_loss = total_loss / len(dataloader)
        print("epoch_loss: ", epoch_loss)
        loss_hist.append(epoch_loss)
        scheduler.step()
        if epoch % 2 == 0 or epoch == epochs - 1:
            # Save checkpoint with meaningful loss value (epoch average)
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": epoch_loss,
            }
            save_path = f"checkpoint{epoch}.pth"
            torch.save(checkpoint, save_path)
            print(f"Saved checkpoint to {save_path} (epoch_loss={epoch_loss:.6f})")

    return loss_hist

In [66]:
train(model,
    train_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=70,
    accumulation_steps=2)

Epoch 1/70: 100%|██████████| 5/5 [00:04<00:00,  1.06it/s]


epoch_loss:  4.1081414222717285
Saved checkpoint to checkpoint0.pth (epoch_loss=4.108141)


Epoch 2/70: 100%|██████████| 5/5 [00:04<00:00,  1.01it/s]


epoch_loss:  2.785554885864258


Epoch 3/70: 100%|██████████| 5/5 [00:04<00:00,  1.07it/s]


...

Epoch 69/70: 100%|██████████| 5/5 [00:04<00:00,  1.07it/s]


epoch_loss:  0.48762404918670654
Saved checkpoint to checkpoint68.pth (epoch_loss=0.487624)


Epoch 70/70: 100%|██████████| 5/5 [00:04<00:00,  1.00it/s]

epoch_loss:  0.4854337900876999
Saved checkpoint to checkpoint69.pth (epoch_loss=0.485434)


[4.1081414222717285,
 2.785554885864258,
 2.523540425300598,
 2.32477753162384,
 2.159827709197998,
 2.039668250083923,
 1.9166428089141845,
 1.8179906845092773,
 1.726658833026886,
 1.6733569502830505,
 1.584722626209259,
 1.5338619709014893,
 1.5195475101470948,
 1.4527328372001649,
 1.4102658033370972,
 1.392804706096649,
 1.3983350038528441,
 1.3731926202774047,
 1.3525038957595825,
 1.338084840774536,
 1.331274390220642,
 1.318325436115265,
 1.2821543216705322,
 1.268850827217102,
 1.2471065282821656,
 1.1898360133171082,
 1.081730842590332,
 1.1053061127662658,
 1.0354288578033448,
 0.9385261058807373,
 0.9039841175079346,
 0.8800559401512146,
 0.8721763730049134,
 0.8322713255882264,
 0.7833075642585754,
 0.752907532453537,
 0.6964613258838653,
 0.685748839378357,
 0.6487541854381561,
 0.6112539082765579,
 0.5754525005817414,
 0.5696893185377121,
 0.5629444628953933,
 0.5605564415454865,
 0.5534163177013397,
 0.5510221004486084,
 0.54820616543293,
 0.5429776459932327,
 0.5394492